<a href="https://colab.research.google.com/github/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/blob/main/AgenticAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Math category

- chiama i tool se c'è un'espressione già fatta. non rielabora le informazioni per creare l'espressione
- il cross encoder non prende la risposta bene in questo caso
- tool da aggiungere:
  - testare quelli aggiunti
- altre llm da provare:
  - deepseek
  - qwen 7b con quantizzazione
  - qwen for math reasoning

In [1]:
from google.colab import userdata
from huggingface_hub import login
import os
import sys
import time

In [2]:
HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)

In [3]:
repo_url = "https://github.com/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi.git"
repo_name = "NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi"

if os.path.exists("../"+repo_name):
    print("Repository already present, update...")
    !git pull
else:
    print("Repository clone...")
    !git clone {repo_url}
    %cd {repo_name}

sys.path.append('/content/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/NLP_assignment_api_client')

from millionaire_client import MillionaireClient, AuthenticationError, GameError

Repository clone...
Cloning into 'NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi'...
remote: Enumerating objects: 343, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 343 (delta 30), reused 24 (delta 24), pack-reused 304 (from 2)
Receiving objects: 100% (343/343), 35.68 MiB | 8.85 MiB/s, done.
Resolving deltas: 100% (166/166), done.
/content/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi


In [4]:
API_URL  = 'http://131.175.15.22:51111/'
USERNAME = 'GliEmbeddingRuspanti'
PASSWORD = 'GliEmbeddingRuspanti'

client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f'Logged in as: {user.username} (role: {user.role})')
except AuthenticationError as e:
    print(f'Login failed: {e}')

Logged in as: GliEmbeddingRuspanti (role: student)


# Model

In [5]:
!pip install -q transformers accelerate bitsandbytes langchain langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.4 MB/s eta 0:00:00


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer


In [7]:
from transformers import BitsAndBytesConfig
import torch
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NF4 è il default consigliato
    bnb_4bit_compute_dtype=torch.float16,  # fp16 ottimale su T4
    bnb_4bit_use_double_quant=True,     # risparmio memoria aggiuntivo
)

In [ ]:
#non riesco a farla funzionare nella mia catena
"""model_name = "Qwen/Qwen2.5-Math-7B-Instruct"
model_finetuned = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype="auto",
            token=HF_TOKEN,
            quantization_config=bnb_config
        )
model_finetuned.eval()  # disabilita dropout, riduce overhead
tokenizer_finetuned = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)"""

config.json:   0%|          | 0.00/658 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/161 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.32k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [8]:
model_name = "Qwen/Qwen2.5-7B-Instruct"
model_qwen = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype="auto",
            token=HF_TOKEN,
            quantization_config=bnb_config
        )
model_qwen.eval()  # disabilita dropout, riduce overhead
tokenizer_qwen = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
model_name="deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
model_deep = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype="auto",
            token=HF_TOKEN,
            quantization_config=bnb_config
        )
model_deep.eval()  # disabilita dropout, riduce overhead
tokenizer_deep = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

In [ ]:
#troppo debole
"""model_name = "Qwen/Qwen2.5-1.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token"""

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

# Agentic

##Tools

In [9]:
!pip install -U ddgs
!pip install -q duckduckgo-search langchain-community

from langchain_community.tools import DuckDuckGoSearchRun


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


/tmp/ipykernel_1112/738134643.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [10]:
import math
from langchain.tools import tool
from sympy import sympify, Eq, solve
from sympy import factorint
from sympy import isprime
import re


In [11]:
@tool
def empty_tool() -> str:
  """
  An empty tool that does nothing. Use in case the other tools are not useful
  """
  return ""

In [12]:
@tool("calculator")
def calculator(expression: str) -> str:
    """
    Performs arithmetic calculations. Evaluate mathematical expressions.
    """
    return str(eval(expression))

In [13]:
@tool
def solve_equation(expressions)-> str:
      """
      Solve a mathematical equation or a system of mathematical equations.
      Set the expression of each equation as a string. Do not specify the symbols.
      If there are more equations send an array of expressions
      """

      expressions = [expressions]

      equations = []
      symbols_set = set()

      variable_names = set()

      for expr in expressions:
          variable_names.update(
              re.findall(r"[a-zA-Z_]\w*", expr)
          )

      symbols_dict = {
          name: symbols(name)
          for name in variable_names
      }

      for expr in expressions:

          if "=" in expr:

              left, right = expr.split("=")

              eq = Eq(
                  eval(left, {}, symbols_dict),
                  eval(right, {}, symbols_dict)
              )

          else:

              eq = Eq(
                  eval(expr, {}, symbols_dict),
                  0
              )

          equations.append(eq)

          symbols_set.update(eq.free_symbols)

      variables = list(symbols_set)

      result = solve(equations, variables)

      return str(result)

In [14]:
search_tool = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """
    Search the web for information.
    """

    return search_tool.invoke(query)

In [15]:
@tool
def percentage_value(
    value: float,
    total: float
) -> str:
    """
    Compute percentage value/total.
    """

    if total == 0:
        return "Division by zero"

    result = (value / total) * 100

    return str(result)

In [16]:
@tool("percentage_calculator")
def percentage_calculator(expression: str) -> str:
    """
    Calculate percentage expressions.

    Examples:
    - "12% of 50"
    - "20 percent of 400"
    - "30 percentage of 20 percentage of 40"
    - "15% of 12% of 80"
    """

    try:
        expr = expression.lower()
        expr = expr.replace("percent","%")
        expr = expr.replace("percentage","%")
        expr = expr.replace( "%", "/100")
        expr = expr.replace( "of", "*")
        result = eval(expr)

        return str(result)

    except Exception as e:

        return f"Error: I could not calculate the percentage"

In [17]:
from sympy import symbols, diff, sympify

@tool
def derivative( expression: str, variable: str = "x") -> str:
    """
    Compute derivative of an expression in the variable specified as a parameter.
    """

    try:
        x = symbols(variable)
        expr = sympify(expression)
        result = diff(expr, x)
        return str(result)

    except Exception as e:

        return f"Error: {e}"

In [18]:
from sympy import integrate

@tool
def integral( expression: str, variable: str = "x") -> str:
    """
    Compute symbolic integral in the variable specified by the parameter.
    """

    try:

        x = symbols(variable)

        expr = sympify(expression)

        result = integrate(expr, x)

        return str(result)

    except Exception as e:

        return f"Error: {e}"

In [19]:
import statistics

@tool
def mean(numbers: list[float]) -> str:
    """
    Compute arithmetic mean of the numbers sent as an argument.
    """

    return str(statistics.mean(numbers))

In [20]:
from sympy import limit

@tool
def compute_limit(expression: str, variable: str, point: float) -> str:
    """
    Compute mathematical limit of the expression in the variable specified by the argument.
    The limit is computed at the point specified by the argument.
    """

    try:

        x = symbols(variable)

        expr = sympify(expression)

        result = limit(expr, x, point)

        return str(result)

    except Exception as e:

        return f"Error: {e}"

In [21]:
@tool
def quadratic_formula(a: float,b: float,c: float) -> str:
    """
    Solve quadratic equation as a*x**2 + b*x + c=0.
    """

    from sympy import symbols, solve

    x = symbols('x')

    result = solve(
        a*x**2 + b*x + c,
        x
    )

    return str(result)

In [22]:
@tool
def factor_integer(number: int) -> str:
    """
    Prime factorization.
    """

    return str(factorint(number))

In [23]:
@tool
def circle_area(radius: float) -> str:
    """
    Compute area of a circle.
    """
    return str(math.pi * radius**2)

In [24]:
@tool
def is_prime(number: int) -> str:
    """
    Check if number is prime.
    """

    return str(isprime(number))

In [25]:
tools = [calculator, solve_equation, web_search, percentage_value, percentage_calculator, derivative,
         integral, mean, compute_limit, quadratic_formula, factor_integer, circle_area, is_prime]

# Let's inspect the tools
for t in tools:
    print("--")
    print(t.name)
    print(t.description)
    print(t.args)

--
calculator
Performs arithmetic calculations. Evaluate mathematical expressions.
{'expression': {'title': 'Expression', 'type': 'string'}}
--
solve_equation
Solve a mathematical equation or a system of mathematical equations.
Set the expression of each equation as a string. Do not specify the symbols.
If there are more equations send an array of expressions
{'expressions': {'title': 'Expressions'}}
--
web_search
Search the web for information.
{'query': {'title': 'Query', 'type': 'string'}}
--
percentage_value
Compute percentage value/total.
{'value': {'title': 'Value', 'type': 'number'}, 'total': {'title': 'Total', 'type': 'number'}}
--
percentage_calculator
Calculate percentage expressions.

Examples:
- "12% of 50"
- "20 percent of 400"
- "30 percentage of 20 percentage of 40"
- "15% of 12% of 80"
{'expression': {'title': 'Expression', 'type': 'string'}}
--
derivative
Compute derivative of an expression in the variable specified as a parameter.
{'expression': {'title': 'Expression'

In [26]:
from langchain_core.tools import render_text_description

rendered_tools = render_text_description(tools)
print(rendered_tools)

calculator(expression: str) -> str - Performs arithmetic calculations. Evaluate mathematical expressions.
solve_equation(expressions) -> str - Solve a mathematical equation or a system of mathematical equations.
Set the expression of each equation as a string. Do not specify the symbols.
If there are more equations send an array of expressions
web_search(query: str) -> str - Search the web for information.
percentage_value(value: float, total: float) -> str - Compute percentage value/total.
percentage_calculator(expression: str) -> str - Calculate percentage expressions.

Examples:
- "12% of 50"
- "20 percent of 400"
- "30 percentage of 20 percentage of 40"
- "15% of 12% of 80"
derivative(expression: str, variable: str = 'x') -> str - Compute derivative of an expression in the variable specified as a parameter.
integral(expression: str, variable: str = 'x') -> str - Compute symbolic integral in the variable specified by the parameter.
mean(numbers: list[float]) -> str - Compute arithme

## llm creation

In [27]:
!pip install langchain_huggingface

In [28]:
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer,pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
"""pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)"""

In [ ]:
pipe_deep = pipeline(
    "text-generation",
    model=model_deep,
    tokenizer=tokenizer_deep,
    max_new_tokens=100,          # sufficiente per rispondere a MCA
    do_sample=False,            # greedy: più veloce e deterministico
    temperature=None,           # obbligatorio quando do_sample=False
    top_p=None,                 # obbligatorio quando do_sample=False
    return_full_text=False,     # ritorna solo la parte generata, non il prompt
    pad_token_id=tokenizer_deep.eos_token_id,  # evita warning su T4
)

In [51]:
pipe_qwen_router = pipeline(
    "text-generation",
    model=model_qwen,
    tokenizer=tokenizer_qwen,
    max_new_tokens=100,          # sufficiente per rispondere a MCA
    do_sample=False,            # greedy: più veloce e deterministico
    temperature=None,           # obbligatorio quando do_sample=False
    top_p=None,                 # obbligatorio quando do_sample=False
   # return_full_text=False,     # ritorna solo la parte generata, non il prompt
    pad_token_id=tokenizer_qwen.eos_token_id,  # evita warning su T4
)

In [61]:
pipe_qwen_answer = pipeline(
    "text-generation",
    model=model_qwen,
    tokenizer=tokenizer_qwen,
    max_new_tokens=200,          # sufficiente per rispondere a MCA
    do_sample=False,            # greedy: più veloce e deterministico
    temperature=None,           # obbligatorio quando do_sample=False
    top_p=None,                 # obbligatorio quando do_sample=False
   # return_full_text=False,     # ritorna solo la parte generata, non il prompt
    pad_token_id=tokenizer_qwen.eos_token_id,  # evita warning su T4
)

In [ ]:
# è per quello finetuned
"""pipe_finetuned = pipeline(
    "text-generation",
    model=model_finetuned,
    tokenizer=tokenizer_finetuned,
    max_new_tokens=100,          # sufficiente per rispondere a MCA
    do_sample=False,            # greedy: più veloce e deterministico
    temperature=None,           # obbligatorio quando do_sample=False
    top_p=None,                 # obbligatorio quando do_sample=False
    return_full_text=False,     # ritorna solo la parte generata, non il prompt
    pad_token_id=tokenizer_finetuned.eos_token_id,  # evita warning su T4
)"""

In [ ]:
"""llm = HuggingFacePipeline(
    pipeline=pipe,
    pipeline_kwargs={
        "max_new_tokens": 150,
        "temperature": 0
    }
)"""

In [53]:
llm_router = HuggingFacePipeline(
    pipeline=pipe_qwen_router,
    pipeline_kwargs={
        "temperature": 0.2
    }
)

In [62]:
llm_answer = HuggingFacePipeline(
    pipeline=pipe_qwen_answer,
    pipeline_kwargs={
        "temperature": 0.2
    }
)

## Example to see if it works

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant."
    ),
    (
        "user",
        "{input}"
    )
])

In [ ]:
chain = prompt | llm

In [ ]:
response = chain.invoke({
    "input": "What are the capitals of Australia, Canada, Brazil and South Africa?"
})
print(response)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


System: You are a helpful assistant.
Human: What are the capitals of Australia, Canada, Brazil and South Africa? Please provide the names in English.

Assistant: The capitals of Australia, Canada, Brazil and South Africa are:
Australia - Canberra
Canada - Ottawa
Brazil - Brasília
South Africa - Pretoria

Note that these are not necessarily the largest cities within each country. For example, Sydney is the most populous city in Australia, while São Paulo is the most populous city in Brazil. Similarly, Toronto is the most populous city in Canada, while Cape Town is the capital but not the most populous city in South Africa. Also, Brasília is the capital of Brazil, but it was established as a new capital to replace Rio de Janeiro after the 1960s military coup d'état which led to the abolition of the federal republic constitution, making Rio de Janeiro an exclusive status as the seat of government for the Federation of Brazil. The current Constitution re-established Rio de Janeiro's role a

### Prompts

In [ ]:
system_prompt_router = f"""\
            You are a correct assistant that has access ONLY to the following tools.
            Here are the names and descriptions for each tool:

            {rendered_tools}

            Your task is to select the BEST tool.
            Given the user input, return only the name and input of the tool to use.
            Choose the best tool among the ones above only.
            It has to be useful for the user request.
            Return your response as a JSON blob with 'name' and 'arguments' keys.

            The `arguments` should be a dictionary, with keys corresponding to the argument names and the values corresponding to the values requested by the user.
            Give this JSON as an answer. Do not answer with your reasoning!
            If there is no useful tool among the ones above, redirect to the tool with name web_search and the query string as an argument.
            """

In [ ]:
from langchain_core.runnables import RunnablePassthrough

prompt_router = ChatPromptTemplate.from_messages(
    [("system", system_prompt_router), ("user", "{input} and the possible options are {options}")]
)


In [ ]:
final_prompt = ChatPromptTemplate.from_template("""
        You are an helpful and correct assistant answering to a MCA question.
        Question:
        {input}

        Possible answers:
        {options}

        Tool used:
        {tool_call}

        Tool output:
        {output}

        Use the tool output to generate the answer.
        Your answer should be: I used the tool 'tool_name' and the correct answer is 'number_of_the_correct_option'.
        Do NOT answer with anything else
        """)

### Helpers for chain

In [ ]:
import json
def extract_json_after_assistant(text: str):

    decoder = json.JSONDecoder()

    candidate = text.split("Assistant:")[-1]

    start = candidate.find("{")

    obj, _ = decoder.raw_decode(candidate[start:])

    return obj

In [ ]:
from langchain_core.runnables import RunnableLambda

custom_parser = RunnableLambda(extract_json_after_assistant)

In [ ]:
from typing import Any, Dict, Optional, TypedDict
from langchain_core.runnables import RunnableConfig


class ToolCallRequest(TypedDict):
    """A typed dict that shows the inputs into the invoke_tool function."""
    name: str
    arguments: Dict[str, Any]


def invoke_tool( x: Dict[str, Any], config: Optional[RunnableConfig] = None ):
    """
    A function that we can use the perform a tool invocation safely.

    Args:
        tool_call_request: a dict that contains the keys name and arguments.
            The name must match the name of a tool that exists.
            The arguments are the arguments to that tool.
        config: This is configuration information that LangChain uses that contains
            things like callbacks, metadata, etc.See LCEL documentation about RunnableConfig.

    Returns:
        output from the requested tool

    Falls back to empty_tool if:
        tool_call is missing
        name is missing
        tool does not exist
        arguments are malformed
    """

    tool_name_to_tool = {
        tool.name: tool
        for tool in tools
    }

    try:

        # recupera tool_call
        tool_call_request = x.get("tool_call", {})

        # recupera nome tool
        name = tool_call_request.get(
            "name",
            "empty_tool"
        )

        # se il tool non esiste -> empty_tool
        if name not in tool_name_to_tool:
            return empty_tool.invoke({}, config=config)

        requested_tool = tool_name_to_tool[name]

        # recupera arguments
        arguments = tool_call_request.get(
            "arguments",
            {}
        )

        # se arguments non è un dict -> empty_tool
        if not isinstance(arguments, dict):
            return empty_tool.invoke({}, config=config)

        # esegui tool richiesto
        return requested_tool.invoke(
            arguments,
            config=config
        )

    except Exception:

        # fallback totale
        return empty_tool.invoke({}, config=config)

### Chain

In [ ]:
router_chain = (
    {
        "input": RunnableLambda(lambda x: x["input"]),

        "tool_call":
            prompt_router
            | llm
            | custom_parser
    }
)

In [ ]:
chain = (
    router_chain
    | RunnablePassthrough.assign(
        output=invoke_tool
    )
    | final_prompt
    | llm
)

In [ ]:
from langchain_huggingface import ChatHuggingFace

# --- QUESTO è il fix: ChatHuggingFace chiama apply_chat_template ---
chat_llm = ChatHuggingFace(llm=llm)

# --- prompt invariato ---
prompt_router = ChatPromptTemplate.from_messages([
    ("system", system_prompt_router),
    ("user", "{input} and the possible options are {options}")
])

# --- parser JSON robusto ---
def extract_json(ai_message) -> dict:
    """
    Estrae il JSON dalla risposta del modello.
    AIMessage ha un campo .content con il testo generato.
    """
    text = ai_message.content if hasattr(ai_message, "content") else str(ai_message)

    try:
        decoder = json.JSONDecoder()
        start = text.find("{")
        if start == -1:
            raise ValueError("No JSON found")
        obj, _ = decoder.raw_decode(text[start:])
        return obj
    except Exception:
        # fallback: nessun tool trovato
        return {"name": "web_search", "arguments": {"query": text[:100]}}

json_parser = RunnableLambda(extract_json)

# --- chain aggiornata ---
router_chain = {
    "input": RunnableLambda(lambda x: x["input"]),
    "options": RunnableLambda(lambda x: x["options"]),
    "tool_call": prompt_router | chat_llm | json_parser,
}
chain = (
    prompt_router |chat_llm
)

In [ ]:
response = chain.invoke({"input": "Which of the following is true about any discrete probability distribution of a random variable X?",
                          "options": {"0" :  "The probability distribution is bell-shaped and symmetric.",
                                      "1" : "The probabilities of all possible values of X must add up to one.",
                                      "2" :  "The expected value of X = np",
                                      "3" :  "The sum of all possible values of X is equal to one."}
                          })
print(response)

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


content="To determine which statement is true about any discrete probability distribution of a random variable \\(X\\), let's analyze each option step by step.\n\n1. **Option 0: The probability distribution is bell-shaped and symmetric.**\n   - This statement is not necessarily true for all discrete probability distributions. A bell-shaped and symmetric distribution is a characteristic of a normal distribution, which is a continuous probability distribution. Discrete probability distributions do not have to be bell-shaped or symmetric. For example, a uniform distribution" additional_kwargs={} response_metadata={} id='lc_run--019e7836-649d-7b21-bf4b-ce547f6aca73-0' tool_calls=[] invalid_tool_calls=[]


# Model

In [63]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

modelSentence = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def pick_by_all_MiniLM(question: str, options: dict):
    labels = list(options.keys())
    texts = [options[label] for label in labels]

    # --- embeddings ---
    question_embedding = modelSentence.encode([question])
    options_embeddings = modelSentence.encode(texts)

    # --- cosine similarities ---
    scores = cosine_similarity(
        question_embedding,
        options_embeddings
    )[0]

    # --- softmax probabilities ---
    exp_scores = np.exp(scores - np.max(scores))
    probs = exp_scores / exp_scores.sum()

    # --- ranking ---
    sorted_idx = np.argsort(scores)[::-1]

    sorted_labels = [labels[i] for i in sorted_idx]
    sorted_scores = scores[sorted_idx]
    sorted_probs = probs[sorted_idx]

    best_label = sorted_labels[0]
    best_score = float(sorted_scores[0])
    best_prob = float(sorted_probs[0])

    second_score = float(sorted_scores[1]) if len(sorted_scores) > 1 else 0.0

    # --- GAP MEDIO (best vs all others) ---
    gap_mean = (
        float(best_score - np.mean(sorted_scores[1:]))
        if len(labels) > 1 else 0.0
    )

    # --- NORMALIZED MARGIN ---
    score_range = np.max(scores) - np.min(scores) + 1e-8
    normalized_margin = (best_score - second_score) / score_range

    # --- probabilistic closeness between top2 ---
    relative_second_closeness = (
        np.exp(second_score) /
        (np.exp(best_score) + np.exp(second_score))
    )

    # --- full option breakdown ---
    options_scores = {
        labels[i]: float(scores[i])
        for i in range(len(labels))
    }

    options_probs = {
        labels[i]: float(probs[i])
        for i in range(len(labels))
    }

    summary = {
        "best_option": best_label,
        "best_score": best_score,
        "best_probability": best_prob,

        "scores": options_scores,
        "softmax_probabilities": options_probs,

        "gap_mean": gap_mean,
        "relative_second_closeness": float(relative_second_closeness),
        "normalized_margin": float(normalized_margin)
    }

    return summary, best_label

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [34]:
import numpy as np
from typing import Callable

In [64]:
class Model:
    """Base class. Subclasses implement generate().
       answer_fn decide how to get the final option."""
    def __init__(self, name: str, answer_fn: Callable):
        self.name = name
        self.answer_fn = answer_fn

    def generate(self, question: str, system_prompt: str = "") -> str:
        raise NotImplementedError

    def answer(self, question: str, options: dict, system_prompt: str = "") -> str:
        raw_output = self.generate(question, options, system_prompt)
        summary_answer, answer = self.answer_fn(raw_output, options)
        print(f"MODEL ANSWER ----->{raw_output}")
        return summary_answer, answer

    def __repr__(self):
        return f"{self.__class__.__name__}(name={self.name!r}, answer_fn={self.answer_fn.__name__!r})"

In [65]:
from typing import Callable, Any, Dict, Optional, TypedDict
import json

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import (
    RunnableLambda,
    RunnablePassthrough,
    RunnableConfig
)
from langchain_core.tools import render_text_description


class AgenticModel(Model):

    def __init__(
        self,
        name: str,
        router_llm,
        final_llm,
        tools,
        answer_fn: Callable
    ):

        super().__init__(name, answer_fn)

        self.router_llm = router_llm
        self.final_llm = final_llm
        self.tools = tools

        # =========================
        # TOOL DESCRIPTIONS
        # =========================

        self.rendered_tools = render_text_description(
            self.tools
        )

        # =========================
        # ROUTER SYSTEM PROMPT
        # =========================
        self.system_prompt_router = f"""\
            You are a correct assistant that has access ONLY to the following tools.
            Here are the names and descriptions for each tool:

            {self.rendered_tools}

            Your task is to select the BEST tool.
            Given the user input, return only the name and input of the tool to use.
            Choose the best tool among the ones above only.
            It has to be useful for the user request.
            Return your response as a JSON blob with 'name' and 'arguments' keys.

            The 'name' should be the name of the tool to use that is part of the ones above.
            The `arguments` should be a dictionary, with keys corresponding to the argument names and the values corresponding to the values requested by the user.
            Give this JSON after `Assistant:`
            If there is no useful tool among the ones above, redirect to the tool with name web_search and the query string as an argument.
            """

        # =========================
        # ROUTER PROMPT
        # =========================

        self.prompt_router = ChatPromptTemplate.from_messages(
            [
                ("system", self.system_prompt_router),
                ("user", "{input} and the possible answers are {options}")
            ]
        )

        # =========================
        # FINAL PROMPT
        # =========================
        self.final_prompt = ChatPromptTemplate.from_template("""
        You are an helpful and correct assistant answering to a MCA question.
        Question:
        {input}

        Possible answers:
        {options}

        Tool used:
        {tool_call}

        Tool output:
        {output}

        Use the tool output to generate the answer.
        Your answer should be: I used the tool 'tool_name' and the correct answer is option 'number_of_the_correct_option': 'text_of_the_correct_option'.
        Do NOT answer with anything else
        """)

        # =========================
        # PARSER
        # =========================

        self.custom_parser = RunnableLambda(
            self.extract_json_after_assistant
        )

        # =========================
        # ROUTER CHAIN
        # =========================
        self.router_chain = (
            {
                "input": RunnableLambda(
                    lambda x: x["input"]
                ),
                "options": RunnableLambda(
                    lambda x: x["options"]
                ),
                "tool_call":
                    self.prompt_router
                    | self.router_llm
                    | self.custom_parser
            }
        )

        # =========================
        # FULL CHAIN
        # =========================

        self.chain = (
            self.router_chain
            | RunnablePassthrough.assign(
                output=self.invoke_tool
            )
            | self.final_prompt
            | self.final_llm
        )

    # ==================================
    # JSON PARSER
    # ==================================
    def extract_json_after_assistant(self, text: str):
        try:

            decoder = json.JSONDecoder()

            candidate = text.split("Assistant:")[-1]

            start = candidate.find("{")

            obj, _ = decoder.raw_decode(candidate[start:])

            return obj

        except Exception:
            return {
                "name": "empty_tool",
                "arguments": {}
            }

    # ==================================
    # TOOL INVOCATION
    # ==================================
    def invoke_tool(
        self,
        x: Dict[str, Any],
        config: Optional[RunnableConfig] = None
    ):
        """
        A function that we can use the perform a tool invocation safely.

        Args:
            tool_call_request: a dict that contains the keys name and arguments.
                The name must match the name of a tool that exists.
                The arguments are the arguments to that tool.
            config: This is configuration information that LangChain uses that contains
                things like callbacks, metadata, etc.See LCEL documentation about RunnableConfig.

        Returns:
            output from the requested tool

        Falls back to empty_tool if:
            tool_call is missing
            name is missing
            tool does not exist
            arguments are malformed
        """
        tool_name_to_tool = {
            tool.name: tool
            for tool in self.tools
        }

        try:
            print(x)
            tool_call_request = x.get(
                "tool_call",
                {}
            )

            name = tool_call_request.get(
                "name",
                "empty_tool"
            )

            if name not in tool_name_to_tool:

                return empty_tool.invoke({})

            requested_tool = tool_name_to_tool[
                name
            ]

            arguments = tool_call_request.get(
                "arguments",
                {}
            )

            if not isinstance(arguments, dict):

                return empty_tool.invoke({})

            return requested_tool.invoke(
                arguments,
                config=config
            )

        except Exception:

            return empty_tool.invoke({})

    # ==================================
    # GENERATE
    # ==================================

    def generate(
        self,
        question: str,
        options: dict,
        system_prompt: str = ""
    ) -> str:

        response = self.chain.invoke(
            {
                "input": question,
                "options": options
            }
        )

        # HuggingFacePipeline può restituire stringa
        # oppure AIMessage

        text = response.content if hasattr(response, "content") else str(response)

        if "Assistant:" in text:
            return text.split("Assistant:")[-1].strip()

        return text.strip()

        #if hasattr(response, "content"):

         #   return response.content

       # return str(response)

# Game

In [66]:
def play_game(game, model_name, verbose=False):
    #models = models_configs["models"]
    #system_prompt = models_configs["system_prompt"]
    agenticModel = AgenticModel(model_name, llm_router, llm_answer, tools, pick_by_all_MiniLM)
    log = []

    while game.in_progress:
        question = game.current_question
        if not question:
            print("No question available. Game may have ended.")
            break

        print(f"\n--- Level {game.current_level} ---")
        print(f"Q: {question.text}")
        for opt in question.options:
            print(f"  [{opt.id}] {opt.text}")

        time_left = game.time_remaining
        if time_left:
            print(f"\nTime remaining: {time_left:.1f}s")

        options = {f"{opt.id}": opt.text for opt in question.options}

        t0 = time.time()
        answer_summary, answer_input = agenticModel.answer(question.text, options)
        #answer_summary, answer_input = answer_ensemble(
         #   models, question.text, options, system_prompt, verbose=verbose
        #)
        inference_time = time.time() - t0

        print(f"Agentic answer: {answer_input}")
        answer_id = int(answer_input)
        choosen_answer = question.options[answer_id]

        result = game.answer(answer_id)

        if result.correct:
            print(" CORRECT!")
            if result.game_over:
                print(f"\n CONGRATULATIONS! You completed the game!")
                print(f" Final earnings: ${result.earned_amount:,.2f}")
            else:
                print(f" Earned so far: ${result.earned_amount:,.2f}")
        elif result.timed_out:
            print("TIMED OUT!")
            print(f"\n Game Over! | Final earnings: ${result.earned_amount:,.2f}")
        elif not result.correct:
            print(" WRONG ANSWER!")
            print(f"\n Game Over! | Final earnings: ${result.earned_amount:,.2f}")
        # save outcome in the log(useful for graphs)
        entry = {
            'level'          : game.current_level,
            'question'       : question.text,
            'options'        : question.options,
            'chosen_option'  : choosen_answer.text,
            'correct'        : result.correct,
            'timed_out'      : result.timed_out,
            'inference_time' : round(inference_time, 2),
            #'answer_summary' : answer_summary,
        }
        log.append(entry)

    #ensemble_name = " + ".join(m.name for m in models)

    summary = {
        'model'          : model_name,
        'final_level'    : game.current_level,
        'earned_amount'  : game.earned_amount,
        'num_questions'  : len(log),
        'num_correct'    : sum(1 for e in log if e['correct']),
        'num_timed_out'  : sum(1 for e in log if e['timed_out']),
        'avg_inference_s': round(sum(e['inference_time'] for e in log) / max(len(log), 1), 2),
        'log'            : log,
    }

    print(f"\n=== Game Summary ===")
    #print(f"Ensemble  : {ensemble_name}")
    print(f"Reached Level: {game.current_level}")
    print(f"Total Earnings: ${game.earned_amount:,.2f}")

    return summary

In [67]:
model_name="agenticModel"
print(f"\n########## MODEL: {model_name} ##########")

    #model = config["model"]
model_results = []
for comp_id in [3]:
    print(f"\n--- Competition {comp_id} ---")

    game = client.game.start(competition_id=comp_id)

    summary = play_game(game, model_name)

    model_results.append(summary)


########## MODEL: agenticModel ##########

--- Competition 3 ---


Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Level 1 ---
Q: What is the greatest common divisor of $2^{1001}-1$ and $2^{1012}-1$?
  [0] 2047
  [1] 2049
  [2] 2048
  [3] 1

Time remaining: 29.9s


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'input': 'What is the greatest common divisor of $2^{1001}-1$ and $2^{1012}-1$?', 'options': {'0': '2047', '1': '2049', '2': '2048', '3': '1'}, 'tool_call': {'query': 'greatest common divisor of 2^1001-1 and 2^1012-1'}}
MODEL ANSWER ----->Human: 
        You are an helpful and correct assistant answering to a MCA question.
        Question:
        What is the greatest common divisor of $2^{1001}-1$ and $2^{1012}-1$?

        Possible answers:
        {'0': '2047', '1': '2049', '2': '2048', '3': '1'}

        Tool used:
        {'query': 'greatest common divisor of 2^1001-1 and 2^1012-1'}

        Tool output:
        

        Use the tool output to generate the answer.
        Your answer should be: I used the tool 'tool_name' and the correct answer is option 'number_of_the_correct_option': 'text_of_the_correct_option'.
        Do NOT answer with anything else
         I used the tool 'query' and the correct answer is option '0': '2047'. 

        The greatest common divisor (GCD) o